# Taller: Flujo de Trabajo de Regresión con Taxi Price Prediction

Este notebook guía un flujo de trabajo realista para entrenar, comparar y validar modelos de regresión en un contexto similar al día a día de ciencia de datos.

Dataset base de referencia: https://www.kaggle.com/datasets/denkuznetz/taxi-price-prediction

## 1. Contexto y objetivo

Se tomó este dataset para practicar ajuste de modelos de **regresión** y evaluación rigurosa del desempeño.

Tu objetivo como estudiante es:
- Cargar correctamente los datos compartidos por el profesor.
- Validar calidad de carga y lectura.
- Realizar análisis exploratorio (EDA) y diagnóstico de calidad de datos.
- Preparar variables numéricas y categóricas.
- Particionar datos de desarrollo en 80/20 (entrenamiento/prueba).
- Ajustar y comparar varios modelos de regresión.
- Elegir el mejor modelo según métricas.
- Evaluar el mejor modelo con el conjunto de validación final (10%).

## 2. Recordatorio importante sobre los datos

Particiones ya preparadas por el profesor:
- Desarrollo 90%: `exercises/clase_20260728/data/splits/taxi_dev_90.csv`
- Validación 10%: `exercises/clase_20260728/data/splits/taxi_validation_10.csv`

Resumen del proceso de partición ya realizado:
- Filas originales: 1000
- Filas con target no nulo usadas para modelado: 951
- Excluidas por Trip_Price nulo: 49
- Desarrollo 90%: 855
- Validación 10%: 96

## 3. Primero: entorno Google Colab

Este notebook está pensado para ejecutarse en **Google Colab**.

Opciones para cargar los CSV:
1. Leerlos directo desde un enlace RAW de GitHub (si el profesor comparte un enlace público).
2. Subir los CSV manualmente a la sesión de Colab y luego leerlos desde `/content`.

Buenas prácticas:
- Trabajar con funciones reutilizables.
- Evitar repetir lógica.
- Separar claramente: carga, EDA, preparación, entrenamiento, evaluación.
- Fijar semilla para reproducibilidad.

In [ ]:
# Si estás en Colab, instala dependencias (descomenta si es necesario).
# !pip -q install scikit-learn pandas numpy matplotlib seaborn

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

RANDOM_STATE = 42
TARGET_COL = "Trip_Price"
pd.set_option("display.max_columns", None)

## 4. Carga de datos (GitHub o subida manual)

Configura una de estas dos estrategias y ejecuta.

In [ ]:
# Opcion A: usar URLs RAW de GitHub (si están disponibles públicamente).
DEV_URL = ""
VAL_URL = ""

# Opcion B: usar rutas locales en Colab tras subir archivos manualmente.
# Ejemplo: /content/taxi_dev_90.csv y /content/taxi_validation_10.csv
DEV_LOCAL_PATH = "/content/taxi_dev_90.csv"
VAL_LOCAL_PATH = "/content/taxi_validation_10.csv"

def load_csv_smart(remote_url: str, local_path: str) -> pd.DataFrame:
    if isinstance(remote_url, str) and remote_url.strip():
        print(f"Leyendo desde URL: {remote_url}")
        return pd.read_csv(remote_url)
    if os.path.exists(local_path):
        print(f"Leyendo desde archivo local: {local_path}")
        return pd.read_csv(local_path)
    raise FileNotFoundError(
        f"No se encontró ni URL ni archivo local válido para: {local_path}"
    )

# Si necesitas subir archivos manualmente en Colab, usa esto:
# from google.colab import files
# files.upload()

dev_df = load_csv_smart(DEV_URL, DEV_LOCAL_PATH)
val_df = load_csv_smart(VAL_URL, VAL_LOCAL_PATH)

print("dev_df shape:", dev_df.shape)
print("val_df shape:", val_df.shape)

## 5. Pruebas de lectura correcta

Antes de modelar, valida que los datos estén bien cargados.

In [ ]:
def quick_data_checks(df: pd.DataFrame, name: str, target_col: str) -> None:
    print(f"\n=== Chequeos de {name} ===")
    print("Forma:", df.shape)
    print("Columnas:", list(df.columns))
    print("Duplicados:", df.duplicated().sum())
    print("Tipos de dato:")
    print(df.dtypes)
    if target_col in df.columns:
        print(f"Nulos en {target_col}:", df[target_col].isna().sum())
    print("Primeras filas:")
    display(df.head())

quick_data_checks(dev_df, "Desarrollo 90%", TARGET_COL)
quick_data_checks(val_df, "Validacion 10%", TARGET_COL)

## 6. EDA descriptivo y diagnóstico

En esta etapa debes:
- Entender la distribución de la variable objetivo.
- Revisar nulos por variable.
- Detectar posibles atípicos.
- Explorar relaciones entre variables y target.

In [ ]:
def profile_missing(df: pd.DataFrame) -> pd.DataFrame:
    miss = df.isna().mean().sort_values(ascending=False).rename("missing_ratio")
    return miss.to_frame()

display(dev_df.describe(include="all"))
display(profile_missing(dev_df).head(15))

plt.figure(figsize=(7, 4))
sns.histplot(dev_df[TARGET_COL].dropna(), kde=True, bins=30)
plt.title("Distribucion de Trip_Price")
plt.show()

In [ ]:
# Detección simple de atípicos con IQR para variables numéricas
def iqr_outlier_summary(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in numeric_cols:
        s = df[col].dropna()
        if s.empty:
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        outliers = ((s < low) | (s > high)).sum()
        rows.append({
            "feature": col,
            "outliers": int(outliers),
            "outlier_ratio": outliers / len(s)
        })
    return pd.DataFrame(rows).sort_values("outlier_ratio", ascending=False)

numeric_columns = dev_df.select_dtypes(include=["number"]).columns.tolist()
numeric_columns_no_target = [c for c in numeric_columns if c != TARGET_COL]
display(iqr_outlier_summary(dev_df, numeric_columns_no_target).head(15))

## 7. Preparación del dataset

Recomendación de buenas prácticas:
- Usar `Pipeline` y `ColumnTransformer` para evitar fuga de información.
- Imputar nulos por tipo de variable.
- Escalar numéricas cuando aplique.
- Codificar categóricas con One-Hot Encoding.

In [ ]:
# Separar features y target
X = dev_df.drop(columns=[TARGET_COL]).copy()
y = dev_df[TARGET_COL].copy()

# Partición interna 80/20 sobre el dataset de desarrollo (90%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

## 8. Entrenamiento de modelos y benchmarking

Se evaluarán varios modelos con métricas de regresión:
- MAE
- RMSE
- R²

Luego construirás una tabla comparativa para elegir el mejor modelo.

In [ ]:
def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

def train_and_evaluate_models(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    preprocessor: ColumnTransformer,
) -> tuple[pd.DataFrame, dict]:
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
        "Lasso": Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000),
        "RandomForest": RandomForestRegressor(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    }

    results = []
    fitted_pipelines = {}

    for name, model in models.items():
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        metrics = regression_metrics(y_test, y_pred)

        results.append({
            "model": name,
            **metrics
        })
        fitted_pipelines[name] = pipe

    bench = pd.DataFrame(results).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
    return bench, fitted_pipelines

benchmark_df, fitted_models = train_and_evaluate_models(
    X_train, X_test, y_train, y_test, preprocessor
)

display(benchmark_df)

In [ ]:
# Selección del mejor modelo según RMSE mínimo
best_model_name = benchmark_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

print("Mejor modelo seleccionado:", best_model_name)
display(benchmark_df)

## 9. Evaluación final sobre validación 10%

Aquí automatizamos una función que reciba como entrada un **modelo ajustado** y evalúe su desempeño en la partición de validación final.

In [ ]:
def evaluate_on_holdout_validation(
    fitted_model: Pipeline,
    validation_df: pd.DataFrame,
    target_col: str = TARGET_COL,
) -> tuple[pd.DataFrame, dict]:
    if target_col not in validation_df.columns:
        raise ValueError(f"No existe la columna target: {target_col}")

    X_val = validation_df.drop(columns=[target_col]).copy()
    y_val = validation_df[target_col].copy()

    y_pred = fitted_model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred)

    preds = validation_df.copy()
    preds["y_pred"] = y_pred
    preds["abs_error"] = (preds[target_col] - preds["y_pred"]).abs()

    return preds, metrics

val_predictions_df, val_metrics = evaluate_on_holdout_validation(
    fitted_model=best_model,
    validation_df=val_df,
    target_col=TARGET_COL,
)

print("Métricas en validación final (10%):")
print(json.dumps(val_metrics, indent=2))
display(val_predictions_df.head())

## 10. Entregables sugeridos

Tu notebook final debería incluir:
1. Carga reproducible de datos (GitHub o archivos en Colab).
2. Evidencia de validación de lectura (shape, tipos, nulos, head).
3. EDA con hallazgos de atípicos y ausentes.
4. Preparación de datos con pipeline modular.
5. Entrenamiento y comparación de múltiples modelos.
6. Tabla benchmarking con MAE, RMSE y R².
7. Selección justificada del mejor modelo.
8. Evaluación final con la función automatizada sobre `taxi_validation_10.csv`.

Cierre esperado: reportar métricas de validación final y conclusiones de negocio/técnicas.